# 카드 그림 만들기 (Colab 무료 GPU)

**순서**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택
2. 아래 칸을 위에서부터 ▶ 로 하나씩 실행 (또는 `런타임 → 모두 실행`)
3. 마지막 칸이 끝나면 `cards_out.zip` 이 자동 다운로드돼요
4. 그 zip 을 채팅에 올려 주세요. 제가 게임에 등록해요

※ 이 그림은 AI가 만든 그림이에요. 사진이 없는 카드(신화·옛 발명품)에만 쓰는 걸 추천해요.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors pillow

## 만들 카드 목록
`id`는 카드 id, `desc`는 그릴 내용(영어). 바꾸거나 줄을 추가하면 돼요.

In [ ]:
CARDS = [
    ("med_opium", "opium poppy flower with seed pods, botanical study"),
    ("myth_siren", "a siren of Greek myth singing on a rocky shore, ship in the distance"),
]
STYLE = ("antique 17th century copperplate engraving, sepia ink on aged parchment, "
         "fine hatching, museum plate illustration, no text, no letters, no watermark")
NEG = "photo, modern, color, text, letters, watermark, signature, frame, blurry, deformed"


In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
pipe.enable_attention_slicing()


In [ ]:
import os, zipfile
from PIL import Image, ImageOps, ImageDraw
os.makedirs("cards_out", exist_ok=True)
for cid, desc in CARDS:
    img = pipe(prompt=f"{desc}, {STYLE}", negative_prompt=NEG, width=1024, height=768,
               num_inference_steps=28, guidance_scale=7).images[0]
    img = ImageOps.fit(img.convert("RGB"), (512, 384))
    d = ImageDraw.Draw(img); d.rectangle([3, 3, 508, 380], outline=(207, 159, 77), width=2)  # 금테
    img.save(f"cards_out/{cid}.png"); print("완료", cid)
with zipfile.ZipFile("cards_out.zip", "w") as z:
    for f in os.listdir("cards_out"): z.write(f"cards_out/{f}", f)
from google.colab import files; files.download("cards_out.zip")
